In [2]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
%pip install PyYAML
import yaml, os

Note: you may need to restart the kernel to use updated packages.


In [3]:
import csv
import os
import shutil

# ========= CONFIG =========
CSV_PATH = r"C:\Users\Lenovo\OneDrive - Hanoi University of Science and Technology\Desktop\HUST\classes.csv"

IMAGE_TRAIN_DIR = r"C:\Users\Lenovo\OneDrive - Hanoi University of Science and Technology\Desktop\HUST\YOLOv11\yolo_dataset\images\train"
IMAGE_VAL_DIR   = r"C:\Users\Lenovo\OneDrive - Hanoi University of Science and Technology\Desktop\HUST\YOLOv11\yolo_dataset\images\val"

OUT_DATASET = r"C:\Users\Lenovo\OneDrive - Hanoi University of Science and Technology\Desktop\HUST\YOLOv11\cls_dataset"

CLASSES = ["A", "B", "C", "D"]
IMAGE_EXTS = (".tiff", ".tif")


# ========= PREPARE OUTPUT FOLDERS =========
for split in ["train", "val"]:
    for cls in CLASSES:
        os.makedirs(os.path.join(OUT_DATASET, split, cls), exist_ok=True)


# ========= INDEX ALL IMAGES =========
image_index = {}

def index_images(root):
    for f in os.listdir(root):
        if f.lower().endswith(IMAGE_EXTS):
            key = f.replace("-Ori-", "-").rsplit(".", 1)[0]
            image_index[key] = os.path.join(root, f)

index_images(IMAGE_TRAIN_DIR)
index_images(IMAGE_VAL_DIR)

print(f"✔ Indexed {len(image_index)} images")


# ========= PARSE CSV (COLUMN-WISE) =========
copied = 0
missing = []

with open(CSV_PATH, newline="", encoding="utf-8") as f:
    reader = csv.reader(f, delimiter=";")
    header = next(reader)  # Train–test | Train–test.1 | Validation

    for row in reader:
        for col_idx, cell in enumerate(row):
            cell = cell.strip().rstrip(",")
            if not cell or "_" not in cell:
                continue

            base, cls = cell.rsplit("_", 1)
            cls = cls.strip().upper()

            if cls not in CLASSES:
                continue

            if base not in image_index:
                missing.append(base)
                continue

            split = "val" if header[col_idx].lower().startswith("validation") else "train"

            src_path = image_index[base]
            dst_path = os.path.join(OUT_DATASET, split, cls, os.path.basename(src_path))

            if not os.path.exists(dst_path):
                shutil.copy2(src_path, dst_path)
                copied += 1


# ========= SUMMARY =========
print("\n🎯 SUMMARY")
print(f"Images copied: {copied}")
print(f"Missing images: {len(set(missing))}")

if missing:
    print("Sample missing:")
    for m in list(set(missing))[:10]:
        print(" ", m)


✔ Indexed 7926 images

🎯 SUMMARY
Images copied: 7254
Missing images: 460
Sample missing:
  Pryntex-17-Ag-04
  Pryntex-01-Ag-07
  Pryntex-14-Ag-09
  Pryntex-51-Ag-04
  Pryntex-28-Ag-01
  Pryntex-32-Ag-09
  Pryntex-50-Ag-09
  Pryntex-10-Ag-06
  Pryntex-05-Ag-09
  Pryntex-30-Ag-06


In [4]:
import os
import random
import shutil
from collections import defaultdict

TRAIN_DIR = r"C:\Users\Lenovo\OneDrive - Hanoi University of Science and Technology\Desktop\HUST\YOLOv11\cls_dataset\train"

TARGET_COUNT = {
    "A": 2000,
    "C": 2000,
    "D": 2000
}

IMAGE_EXTS = (".tif", ".tiff")

random.seed(42)


for cls, target in TARGET_COUNT.items():
    cls_dir = os.path.join(TRAIN_DIR, cls)
    images = [f for f in os.listdir(cls_dir) if f.lower().endswith(IMAGE_EXTS)]

    cur = len(images)
    if cur >= target:
        print(f"{cls}: already {cur}, skip")
        continue

    need = target - cur
    print(f"{cls}: {cur} → {target} (add {need})")

    # phân bố đều số lần copy cho mỗi ảnh
    usage = defaultdict(int)

    for i in range(need):
        src = min(images, key=lambda x: usage[x])
        usage[src] += 1

        name, ext = os.path.splitext(src)
        new_name = f"{name}_os{usage[src]}{ext}"

        shutil.copy2(
            os.path.join(cls_dir, src),
            os.path.join(cls_dir, new_name)
        )


A: 316 → 2000 (add 1684)
C: 780 → 2000 (add 1220)
D: 1209 → 2000 (add 791)


In [5]:
import os

base = r"C:\Users\Lenovo\OneDrive - Hanoi University of Science and Technology\Desktop\HUST\YOLOv11\cls_dataset"

for split in ["train", "val"]:
    print(f"\n{split.upper()}")
    for cls in ["A", "B", "C", "D"]:
        path = os.path.join(base, split, cls)
        print(cls, len(os.listdir(path)))



TRAIN
A 2000
B 2948
C 2000
D 2000

VAL
A 264
B 1244
C 116
D 377


In [6]:
from PIL import Image
import os

ROOT = r"C:\Users\Lenovo\OneDrive - Hanoi University of Science and Technology\Desktop\HUST\YOLOv11\cls_dataset\train"
OUT = r"C:\Users\Lenovo\OneDrive - Hanoi University of Science and Technology\Desktop\HUST\YOLOv11\cls_dataset_resized\train"
SIZE = (224, 224)

for cls in os.listdir(ROOT):
    os.makedirs(os.path.join(OUT, cls), exist_ok=True)
    for f in os.listdir(os.path.join(ROOT, cls)):
        img = Image.open(os.path.join(ROOT, cls, f)).convert("RGB")
        img = img.resize(SIZE, Image.BILINEAR)
        img.save(os.path.join(OUT, cls, f))

VAL_ROOT = r"C:\Users\Lenovo\OneDrive - Hanoi University of Science and Technology\Desktop\HUST\YOLOv11\cls_dataset\val"
VAL_OUT = r"C:\Users\Lenovo\OneDrive - Hanoi University of Science and Technology\Desktop\HUST\YOLOv11\cls_dataset_resized\val"

for cls in os.listdir(VAL_ROOT):
    os.makedirs(os.path.join(VAL_OUT, cls), exist_ok=True)
    for f in os.listdir(os.path.join(VAL_ROOT, cls)):
        img = Image.open(os.path.join(VAL_ROOT, cls, f)).convert("RGB")
        img = img.resize(SIZE, Image.BILINEAR)
        img.save(os.path.join(VAL_OUT, cls, f))

In [2]:
import os
from ultralytics import YOLO

# Load model
model = YOLO(r"C:\Users\Lenovo\runs\classify\train2\weights\best.pt")

val_dir = r"cls_dataset_resized\val"
classes = sorted(os.listdir(val_dir))

wrong = []

for cls in classes:
    folder = os.path.join(val_dir, cls)
    for f in os.listdir(folder):
        path = os.path.join(folder, f)

        result = model(path, verbose=False)[0]
        pred_class = classes[int(result.probs.top1)]

        if pred_class != cls:
            wrong.append((f, cls, pred_class))

print("Total wrong:", len(wrong))
print("Sample wrong predictions:")
print(wrong[:20])

Total wrong: 744
Sample wrong predictions:
[('762-27-Ori-Ag-109.tif', 'A', 'B'), ('762-27-Ori-Ag-14.tif', 'A', 'B'), ('762-27-Ori-Ag-140.tif', 'A', 'B'), ('762-27-Ori-Ag-143.tif', 'A', 'B'), ('762-27-Ori-Ag-145.tif', 'A', 'B'), ('762-27-Ori-Ag-191.tif', 'A', 'B'), ('762-27-Ori-Ag-197.tif', 'A', 'B'), ('762-27-Ori-Ag-25.tif', 'A', 'B'), ('762-27-Ori-Ag-77.tif', 'A', 'B'), ('762-27-Ori-Ag-78.tif', 'A', 'B'), ('762-27-Ori-Ag-83.tif', 'A', 'B'), ('762-28-Ori-Ag-16.tif', 'A', 'B'), ('762-28-Ori-Ag-161.tif', 'A', 'B'), ('762-28-Ori-Ag-35.tif', 'A', 'B'), ('762-28-Ori-Ag-73.tif', 'A', 'B'), ('762-28-Ori-Ag-74.tif', 'A', 'B'), ('762-29-Ori-Ag-112.tif', 'A', 'B'), ('762-29-Ori-Ag-125.tif', 'A', 'B'), ('762-29-Ori-Ag-132.tif', 'A', 'B'), ('762-29-Ori-Ag-142.tif', 'A', 'B')]


In [5]:
for f in os.listdir(os.path.join(val_dir, "B")):
    path = os.path.join(val_dir, "B", f)

    result = model(path, verbose=False)[0]
    pred_class = classes[int(result.probs.top1)]

    if pred_class == "C":
        print(f)

762-27-Ori-Ag-135.tif
762-27-Ori-Ag-187.tif
762-27-Ori-Ag-194.tif
762-27-Ori-Ag-51.tif
762-27-Ori-Ag-98.tif
762-28-Ori-Ag-122.tif
762-28-Ori-Ag-128.tif
762-28-Ori-Ag-147.tif
762-28-Ori-Ag-149.tif
762-28-Ori-Ag-170.tif
762-28-Ori-Ag-178.tif
762-28-Ori-Ag-181.tif
762-28-Ori-Ag-44.tif
762-28-Ori-Ag-46.tif
762-29-Ori-Ag-155.tif
762-29-Ori-Ag-30.tif
762-29-Ori-Ag-45.tif
762-29-Ori-Ag-66.tif
762-29-Ori-Ag-99.tif
762-30-Ori-Ag-12.tif
762-30-Ori-Ag-121.tif
762-30-Ori-Ag-125.tif
762-30-Ori-Ag-27.tif
762-30-Ori-Ag-28.tif
762-30-Ori-Ag-34.tif
762-30-Ori-Ag-47.tif
762-31-Ori-Ag-107.tif
762-31-Ori-Ag-145.tif
762-31-Ori-Ag-17.tif
762-32-Ori-Ag-106.tif
762-32-Ori-Ag-11.tif
762-32-Ori-Ag-144.tif
762-32-Ori-Ag-153.tif
762-32-Ori-Ag-166.tif
762-32-Ori-Ag-194.tif
762-32-Ori-Ag-6.tif
762-32-Ori-Ag-72.tif
762-32-Ori-Ag-73.tif
762-32-Ori-Ag-87.tif
762-33-Ori-Ag-152.tif
762-33-Ori-Ag-161.tif
762-33-Ori-Ag-175.tif
762-33-Ori-Ag-183.tif
762-33-Ori-Ag-187.tif
762-33-Ori-Ag-202.tif
762-33-Ori-Ag-57.tif
762-34-Or

In [4]:
import os
import numpy as np
from collections import defaultdict
from ultralytics import YOLO

# Load model
model = YOLO(r"C:\Users\Lenovo\runs\classify\train2\weights\best.pt")

val_dir = r"cls_dataset_resized\val"
classes = sorted(os.listdir(val_dir))

# Tạo ma trận nhầm lẫn
conf_matrix = np.zeros((len(classes), len(classes)), dtype=int)

for true_idx, true_cls in enumerate(classes):
    folder = os.path.join(val_dir, true_cls)
    for f in os.listdir(folder):
        path = os.path.join(folder, f)

        result = model(path, verbose=False)[0]
        pred_idx = int(result.probs.top1)

        conf_matrix[true_idx][pred_idx] += 1

# In confusion matrix
print("Confusion Matrix (rows = True, cols = Pred):\n")
print(classes)
print(conf_matrix)

# Tìm các cặp nhầm nhiều nhất (không tính đúng lớp)
confusions = []

for i in range(len(classes)):
    for j in range(len(classes)):
        if i != j and conf_matrix[i][j] > 0:
            confusions.append((classes[i], classes[j], conf_matrix[i][j]))

# Sắp xếp theo số lượng giảm dần
confusions.sort(key=lambda x: x[2], reverse=True)

print("\nTop Confusions:")
for c in confusions[:10]:
    print(f"{c[0]} → {c[1]} : {c[2]} images")

Confusion Matrix (rows = True, cols = Pred):

['A', 'B', 'C', 'D']
[[214  50   0   0]
 [312 869  60   3]
 [  1  26  84   5]
 [  5 226  56  90]]

Top Confusions:
B → A : 312 images
D → B : 226 images
B → C : 60 images
D → C : 56 images
A → B : 50 images
C → B : 26 images
C → D : 5 images
D → A : 5 images
B → D : 3 images
C → A : 1 images
